# 05 — Cross-Gemeinde-Evaluation

Dieses Notebook ist der **zentrale Evaluationsschritt**: Es laedt alle 20 trainierten Modelle und evaluiert jedes auf allen 18 Gemeinden des Datensatzes.

## Ergebnis: 360 Zeilen
20 Modelle (4 Typen x 5 Datenvarianten) x 18 Gemeinden = **360 Evaluationsergebnisse**

## Warum Cross-Gemeinde-Evaluation?
Die Test-Metriken aus dem Training (Skripte 01–04) messen nur die Performance auf der **eigenen** Datenvariante. Die Cross-Evaluation zeigt, wie gut ein Modell auf **fremden** Gemeinden generalisiert:

- **Bekannte Gemeinden**: Im Trainingsset enthalten → niedrige MAE erwartet
- **Unbekannte Gemeinden**: Nicht im Training → Generalisierungstest
- **Generalisierungsluecke**: Differenz (unbekannt - bekannt) in Sekunden

## Wichtiger Hinweis zu Label-Encodings
Die Label-Encodings (from_stop_enc, to_stop_enc, etc.) werden **pro Gemeinde neu berechnet**. Das bedeutet:
- Code 5 in GM0047 ist eine andere Haltestelle als Code 5 in GM0312
- Baummodelle (XGBoost, RF) splitten auf diesen Codes → lernen gemeinde-spezifische Muster
- Bei neuen Gemeinden sind diese Muster nutzlos → schlechte Generalisierung
- Ridge ist davon weniger betroffen, da es nur lineare Beziehungen lernt

**Laufzeit**: ca. 90 Minuten

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from torch.utils.data import DataLoader, TensorDataset

SCRIPT_START = time.time()

In [ ]:
BASE_DIR = Path(".").resolve().parent.parent  # -> Projekt-Root
DATEN_DIR = BASE_DIR / "Daten"
SCRIPT_DIR = Path(".").resolve().parent
MODEL_DIR = SCRIPT_DIR / "models"
OUTPUT_DIR = SCRIPT_DIR / "ergebnisse"
DATA_DIR = SCRIPT_DIR / "data"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 4096 if torch.cuda.is_available() else 1024

# Alle 18 Gemeinden dynamisch ermitteln
GEMEINDEN = sorted([f.stem for f in (DATEN_DIR / "travel_times").glob("GM*.csv")])
TARGET = "travel_time"

print(f"Device: {DEVICE}")
print(f"Gemeinden: {len(GEMEINDEN)} — {GEMEINDEN}")

In [ ]:
# Feature-Definition (identisch zu 00_prepare_data.py)
FEATURE_COLS = (
    ["hour_sin", "hour_cos"]
    + [f"weekday_{d}" for d in range(1, 7)]
    + ["is_weekend", "is_rush_hour"]
    + ["month_sin", "month_cos"]
    + ["segment_dist_m", "from_stop_enc", "to_stop_enc", "route_enc", "line_enc"]
    + ["dwell_time", "seg_position"]
    + ["travel_time_prev"]
)
CAT_COLS = ["from_stop_enc", "to_stop_enc", "route_enc", "line_enc"]

## MLPv2-Modellklasse

Die Architektur muss identisch zum Training sein, damit die gespeicherten Gewichte geladen werden koennen.

In [ ]:
def emb_dim(n_cat: int) -> int:
    return min(16, max(4, int(n_cat ** 0.5)))


class EmbeddingMLP(nn.Module):
    def __init__(self, n_cont: int, vocab_sizes: list[int], emb_dims: list[int]):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(vs, dim) for vs, dim in zip(vocab_sizes, emb_dims)
        ])
        n_input = n_cont + sum(emb_dims)
        self.net = nn.Sequential(
            nn.Linear(n_input, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x_cont, x_cat):
        embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        x = torch.cat([x_cont] + embs, dim=1)
        return self.net(x).squeeze(-1)

## Hilfsfunktionen

Die Datenaufbereitung pro Gemeinde ist identisch zu `00_prepare_data.py`, wird aber hier fuer jede Gemeinde **einzeln** durchgefuehrt (max. 500k Zeilen pro Gemeinde als Kompromiss zwischen Geschwindigkeit und Genauigkeit).

In [ ]:
def parse_points_vectorized(series):
    extracted = series.str.extract(r"POINT\(([^ ]+) ([^ ]+)\)")
    return extracted[0].astype(float).values, extracted[1].astype(float).values


def haversine_vec(lon1, lat1, lon2, lat2):
    R = 6_371_000
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlam / 2) ** 2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))


def mape(y_true, y_pred):
    mask = y_true != 0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


MAX_ROWS = 500_000


def prepare_gemeinde(name: str) -> pd.DataFrame | None:
    """
    Bereitet eine Gemeinde mit der v2-Feature-Pipeline auf.
    Label-Encoding erfolgt pro Gemeinde (NICHT konsistent mit dem Training!).
    """
    tt_path = DATEN_DIR / "travel_times" / f"{name}.csv"
    dt_path = DATEN_DIR / "dwell_times" / f"{name}.csv"
    if not tt_path.exists() or not dt_path.exists():
        return None

    df_tt = pd.read_csv(tt_path, nrows=MAX_ROWS)
    df_dt = pd.read_csv(dt_path, nrows=MAX_ROWS)

    df_tt["from_time"] = pd.to_datetime(df_tt["from_time"])
    df_tt["to_time"] = pd.to_datetime(df_tt["to_time"])
    df_tt["travel_time"] = (df_tt["to_time"] - df_tt["from_time"]).dt.total_seconds()

    df_dt["from_time"] = pd.to_datetime(df_dt["from_time"])
    df_dt["to_time"] = pd.to_datetime(df_dt["to_time"])
    df_dt["dwell_time"] = (df_dt["to_time"] - df_dt["from_time"]).dt.total_seconds()

    df_tt["route"] = df_tt["route"].astype("int64")
    df_dt["route"] = df_dt["route"].astype("int64")

    df = pd.merge(
        df_tt, df_dt[["date", "trip", "route", "stop", "dwell_time"]],
        how="left",
        left_on=["date", "trip", "route", "from_stop"],
        right_on=["date", "trip", "route", "stop"],
    ).drop(columns=["stop"])

    mask_tt = (df["travel_time"] > 0) & (df["travel_time"] <= 600)
    df.loc[df["dwell_time"] > 300, "dwell_time"] = np.nan

    lon_from, lat_from = parse_points_vectorized(df["from_geometry"])
    lon_to, lat_to = parse_points_vectorized(df["to_geometry"])
    df["segment_dist_m"] = haversine_vec(lon_from, lat_from, lon_to, lat_to)
    df["speed_kmh"] = (df["segment_dist_m"] / df["travel_time"].replace(0, np.nan)) * 3.6
    mask_speed = df["speed_kmh"].fillna(0) <= 100
    df = df[mask_tt & mask_speed].copy()
    df.drop(columns=["speed_kmh"], inplace=True)

    if len(df) < 100:
        return None

    # Feature Engineering
    df["date_dt"] = pd.to_datetime(df["date"])
    hour = df["from_time"].dt.hour
    df["hour_sin"] = np.sin(hour * 2 * np.pi / 24)
    df["hour_cos"] = np.cos(hour * 2 * np.pi / 24)

    weekday = df["date_dt"].dt.dayofweek
    for d in range(1, 7):
        df[f"weekday_{d}"] = (weekday == d).astype(int)
    df["is_weekend"] = (weekday >= 5).astype(int)
    df["is_rush_hour"] = (((hour >= 7) & (hour <= 9)) | ((hour >= 15) & (hour <= 18))).astype(int)

    month = df["date_dt"].dt.month
    df["month_sin"] = np.sin(month * 2 * np.pi / 12)
    df["month_cos"] = np.cos(month * 2 * np.pi / 12)

    for col, new_col in [("from_stop", "from_stop_enc"), ("to_stop", "to_stop_enc"),
                          ("route", "route_enc"), ("line", "line_enc")]:
        df[new_col] = df[col].astype("category").cat.codes

    df["dwell_time"] = df["dwell_time"].fillna(0.0)
    df = df.sort_values(["date", "trip", "from_time"])
    df["seg_position"] = df.groupby(["date", "trip"]).cumcount()
    df["travel_time_prev"] = df.groupby(["date", "trip"])["travel_time"].shift(1)
    df["travel_time_prev"] = df["travel_time_prev"].fillna(0.0)

    return df[FEATURE_COLS + [TARGET]].copy()


def evaluate_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": root_mean_squared_error(y_true, y_pred),
        "MAPE": mape(y_true, y_pred),
    }


@torch.no_grad()
def predict_mlpv2(model, X_cont, X_cat, y):
    """MLPv2-Vorhersage mit DataLoader (num_workers=0 fuer Windows)."""
    model.eval()
    ds = TensorDataset(
        torch.tensor(X_cont, dtype=torch.float32),
        torch.tensor(X_cat, dtype=torch.long),
        torch.tensor(y, dtype=torch.float32),
    )
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    preds = []
    for X_c_b, X_cat_b, _ in loader:
        preds.append(model(X_c_b.to(DEVICE), X_cat_b.to(DEVICE)).cpu().numpy())
    return np.concatenate(preds)

## Modelle laden

20 Modelle aus 4 Typen:
- **XGBoost** (5): sklearn-kompatibel, keine Skalierung noetig
- **Random Forest** (5): sklearn, keine Skalierung noetig
- **Ridge** (5): Benoetigt den gespeicherten StandardScaler
- **MLPv2** (5): Benoetigt Scaler + Config fuer Architektur-Rekonstruktion

In [ ]:
models = {}

for variant in ["D1_single", "D2_multi4", "D3_mittel6", "D4_gross10", "D5_fremd"]:
    # XGBoost
    path = MODEL_DIR / f"xgboost_{variant}.joblib"
    if path.exists():
        models[f"XGB_{variant}"] = {"type": "sklearn", "model": joblib.load(path)}

    # Random Forest
    path = MODEL_DIR / f"rf_{variant}.joblib"
    if path.exists():
        models[f"RF_{variant}"] = {"type": "sklearn", "model": joblib.load(path)}

    # Ridge (mit Scaler)
    path = MODEL_DIR / f"ridge_{variant}.joblib"
    if path.exists():
        scaler = joblib.load(MODEL_DIR / f"ridge_scaler_{variant}.joblib")
        models[f"Ridge_{variant}"] = {"type": "sklearn_scaled", "model": joblib.load(path), "scaler": scaler}

    # MLPv2 (mit Scaler + Config)
    pt_path = MODEL_DIR / f"mlpv2_{variant}.pt"
    cfg_path = MODEL_DIR / f"mlpv2_config_{variant}.joblib"
    scl_path = MODEL_DIR / f"mlpv2_scaler_{variant}.joblib"
    if pt_path.exists() and cfg_path.exists() and scl_path.exists():
        config = joblib.load(cfg_path)
        scaler = joblib.load(scl_path)
        model = EmbeddingMLP(config["n_cont"], config["vocab_sizes"], config["emb_dims"]).to(DEVICE)
        model.load_state_dict(torch.load(pt_path, map_location=DEVICE, weights_only=True))
        models[f"MLPv2_{variant}"] = {"type": "mlpv2", "model": model, "scaler": scaler, "config": config}

print(f"{len(models)} Modelle geladen.")

## Evaluation: Jedes Modell auf jeder Gemeinde

Pro Gemeinde:
1. Rohdaten laden und aufbereiten (Feature Engineering)
2. Alle 20 Modelle evaluieren (MAE, RMSE, MAPE)
3. Bestes Modell fuer diese Gemeinde anzeigen

In [ ]:
all_results = []

for gm in GEMEINDEN:
    print(f"[{gm}] ", end="", flush=True)
    df = prepare_gemeinde(gm)
    if df is None:
        print("uebersprungen")
        continue

    X = df[FEATURE_COLS]
    y = df[TARGET].values
    n_seg = len(df)
    print(f"{n_seg:>9,} seg  ", end="", flush=True)

    cont_cols = [c for c in FEATURE_COLS if c not in CAT_COLS]
    cat_cols = [c for c in CAT_COLS if c in FEATURE_COLS]

    for model_name, m in models.items():
        if m["type"] == "sklearn":
            y_pred = m["model"].predict(X)
        elif m["type"] == "sklearn_scaled":
            X_scaled = m["scaler"].transform(X)
            y_pred = m["model"].predict(X_scaled)
        elif m["type"] == "mlpv2":
            X_cont = m["scaler"].transform(X[cont_cols])
            X_cat = X[cat_cols].values.astype(np.int64)
            # Kategorische Indices clippen (neue Gemeinden haben evtl. mehr Kategorien)
            for i, col in enumerate(cat_cols):
                max_idx = m["config"]["vocab_sizes"][i] - 1
                X_cat[:, i] = np.clip(X_cat[:, i], 0, max_idx)
            y_pred = predict_mlpv2(m["model"], X_cont, X_cat, y)

        metrics = evaluate_metrics(y, y_pred)
        all_results.append({"gemeinde": gm, "modell": model_name, "n_segmente": n_seg, **metrics})

    # Bestes Modell anzeigen
    gm_results = [r for r in all_results if r["gemeinde"] == gm]
    best = min(gm_results, key=lambda r: r["MAE"])
    print(f"best: {best['modell']} MAE={best['MAE']:.1f}s")

print(f"\nGesamtlaufzeit: {time.time() - SCRIPT_START:.1f}s")

## Ergebnisse speichern und analysieren

In [ ]:
results_df = pd.DataFrame(all_results)
results_df_export = results_df.copy()
results_df_export["MAE"] = results_df_export["MAE"].round(2)
results_df_export["RMSE"] = results_df_export["RMSE"].round(2)
results_df_export["MAPE"] = results_df_export["MAPE"].round(1)
results_df_export.to_csv(OUTPUT_DIR / "cross_evaluation.csv", index=False)

print("Mittlere MAE ueber alle 18 Gemeinden (Ranking):")
means = results_df.groupby("modell")["MAE"].mean().sort_values()
for model_name, mae_val in means.items():
    print(f"  {model_name:<25s}  MAE={mae_val:.2f}s")

## Bekannt vs. Unbekannt

Diese Analyse zeigt die **Generalisierungsluecke** pro Modell: Wie viel schlechter ist die MAE auf unbekannten Gemeinden im Vergleich zu bekannten?

In [ ]:
known_sets = {
    "D1_single": {"GM0047"},
    "D2_multi4": {"GM0047", "GM0059", "GM0281", "GM0590"},
    "D3_mittel6": {"GM0047", "GM0312", "GM0590", "GM0546", "GM0629", "GM1681"},
    "D4_gross10": {"GM0047", "GM0312", "GM0590", "GM0546", "GM0629", "GM1681",
                   "GM0281", "GM1950", "GM1969", "GM1930"},
    "D5_fremd": {"GM0312", "GM0590", "GM0546", "GM0629", "GM1681"},
}

print(f"{'Modell':<25s}  {'bekannt':>10s}  {'unbekannt':>10s}  {'Diff':>8s}")
print(f"{'-'*55}")

for model_name in means.index:
    dv = None
    for k in known_sets:
        if k in model_name:
            dv = k
            break
    if dv is None:
        continue

    known = known_sets[dv]
    m_results = results_df[results_df["modell"] == model_name]
    k_mae = m_results[m_results["gemeinde"].isin(known)]["MAE"].mean()
    u_mae = m_results[~m_results["gemeinde"].isin(known)]["MAE"].mean()
    diff = u_mae - k_mae
    print(f"  {model_name:<25s}  {k_mae:>9.2f}s  {u_mae:>9.2f}s  {diff:>+7.2f}s")